# Player Status and News Notebook

This notebook is an exploratory workflow for building reliable player-availability signals from stats.mlb.com

In [45]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import importlib.util
import json
import urllib.parse
import urllib.request
import urllib.error

import pandas as pd


## Configuration


In [46]:
SEASON = 2023 #date.today().year
LOOKBACK_DAYS = 14
SPORT_ID = 1

START_DATE = date(SEASON - 1, 1, 1)
END_DATE = date.today()
SNAPSHOT_DATE = END_DATE
DATE_TAG = SNAPSHOT_DATE.isoformat().replace("-", "")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://statsapi.mlb.com/api/v1"


## Request helpers (debug-friendly)

These helpers preserve URL, HTTP status, and an error preview so call-shape problems are easy to diagnose.


In [47]:
def build_url(path: str, params: dict | None = None) -> str:
    params = params or {}
    query = urllib.parse.urlencode(params)
    url = f"{BASE_URL}{path}"
    return f"{url}?{query}" if query else url


def fetch_json_debug(path: str, params: dict | None = None, timeout: int = 60) -> dict:
    url = build_url(path, params)
    result = {
        "ok": False,
        "url": url,
        "status": None,
        "error": None,
        "payload": None,
        "response_text_preview": None,
    }

    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            result["status"] = response.status
            raw_text = response.read().decode("utf-8")
            result["response_text_preview"] = raw_text[:500]
            result["payload"] = json.loads(raw_text)
            result["ok"] = True
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        result["status"] = exc.code
        result["error"] = f"HTTPError: {exc}"
        result["response_text_preview"] = body[:500]
    except urllib.error.URLError as exc:
        result["error"] = f"URLError: {exc}"
    except json.JSONDecodeError as exc:
        result["error"] = f"JSONDecodeError: {exc}"

    return result


def payload_records(payload: dict | None, keys: list[str]) -> list[dict]:
    if not payload:
        return []
    for key in keys:
        if key in payload and isinstance(payload[key], list):
            return payload[key]
    return []


## Step 1: connectivity sanity check


In [48]:
sports_check = fetch_json_debug("/sports", {"sportId": SPORT_ID})

print("OK:", sports_check["ok"])
print("Status:", sports_check["status"])
print("URL:", sports_check["url"])
print("Error:", sports_check["error"])
print("Preview:", sports_check["response_text_preview"])


OK: True
Status: 200
URL: https://statsapi.mlb.com/api/v1/sports?sportId=1
Error: None
Preview: {"copyright":"Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt","sports":[{"id":1,"code":"mlb","link":"/api/v1/sports/1","name":"Major League Baseball","abbreviation":"MLB","sortOrder":11,"activeStatus":true}]}


## Step 2: endpoint options to evaluate

Use this as a shortlist of endpoints that can contribute to injury / availability status.


In [49]:
endpoint_catalog = pd.DataFrame([
    {
        "purpose": "Historical movement log",
        "endpoint": "/transactions",
        "example_params": "sportId, startDate, endDate, transactionTypes",
        "strength": "Best historical source when type codes are recognized",
        "limitation": "May miss statuses if wrong type codes or if event is not represented as a transaction"
    },
    {
        "purpose": "Discover valid transaction codes",
        "endpoint": "/transactionTypes",
        "example_params": "sportId",
        "strength": "Lets you verify exact server-supported type codes/descriptions",
        "limitation": "Reference metadata only, not player events"
    },
    {
        "purpose": "Live injured list snapshot",
        "endpoint": "/teams/{teamId}/roster",
        "example_params": "rosterType=injured&date=YYYY-MM-DD",
        "strength": "Direct current IL membership for a team/date",
        "limitation": "Snapshot-oriented; historical backfill requires repeated date pulls"
    },
    {
        "purpose": "Live active/40-man context",
        "endpoint": "/teams/{teamId}/roster",
        "example_params": "rosterType=active or 40Man",
        "strength": "Compares injured vs active vs 40-man status",
        "limitation": "Not a transaction history by itself"
    },
    {
        "purpose": "Player-level roster context",
        "endpoint": "/people/{personId}",
        "example_params": "hydrate=currentTeam,rosterEntries(team)",
        "strength": "Adds player context and roster-entry metadata",
        "limitation": "Not a complete replacement for transactions"
    },
])

endpoint_catalog


,purpose,endpoint,example_params,strength,limitation
0,Historical movement log,/transactions,"sportId, startDate, endDate, transactionTypes",Best historical source when type codes are rec...,May miss statuses if wrong type codes or if ev...
1,Discover valid transaction codes,/transactionTypes,sportId,Lets you verify exact server-supported type co...,"Reference metadata only, not player events"
2,Live injured list snapshot,/teams/{teamId}/roster,rosterType=injured&date=YYYY-MM-DD,Direct current IL membership for a team/date,Snapshot-oriented; historical backfill require...
3,Live active/40-man context,/teams/{teamId}/roster,rosterType=active or 40Man,Compares injured vs active vs 40-man status,Not a transaction history by itself
4,Player-level roster context,/people/{personId},"hydrate=currentTeam,rosterEntries(team)",Adds player context and roster-entry metadata,Not a complete replacement for transactions


## Step 4: transaction probes

Use only an unfiltered transaction pull (`type_code=ALL` behavior) because explicit typed probes excluded many real-world transaction categories in testing.


In [51]:
common_params = {
    "sportId": SPORT_ID,
    "startDate": START_DATE.isoformat(),
    "endDate": END_DATE.isoformat(),
}

unfiltered_transactions = fetch_json_debug("/transactions", common_params)

unfiltered_records = payload_records(unfiltered_transactions.get("payload"), ["transactions"])

call_log = pd.DataFrame([
    {
        "probe": "unfiltered",
        "type_code": "ALL",
        "ok": unfiltered_transactions["ok"],
        "status": unfiltered_transactions["status"],
        "row_count": len(unfiltered_records),
        "url": unfiltered_transactions["url"],
        "error": unfiltered_transactions["error"],
    }
])

call_log


,probe,type_code,ok,status,row_count,url,error
0,unfiltered,ALL,True,200,73999,https://statsapi.mlb.com/api/v1/transactions?s...,None


In [ ]:
frames = []

if unfiltered_records:
    base_frame = pd.json_normalize(unfiltered_records)
    base_frame["requested_type_code"] = "ALL"
    frames.append(base_frame)

transactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
transactions = transactions.drop_duplicates() if not transactions.empty else transactions

print(f"Loaded {len(transactions):,} unique rows from unfiltered transaction probe.")

In [60]:
transactions.sample(10)

,id,transaction_date,effective_date,resolutionDate,type_code,type_desc,description,player_id,player_name,person.link,to_team_id,to_team_name,toTeam.link,from_team_id,from_team_name,fromTeam.link,requested_type_code
44290,761887,2024-04-09,2024-04-09,2024-04-09,ASG,Assigned,Kansas City Royals sent 2B Michael Massey on a...,686681.0,Michael Massey,/api/v1/people/686681,541,Omaha Storm Chasers,/api/v1/teams/541,118.0,Kansas City Royals,/api/v1/teams/118,ALL
56520,816770,2025-02-23,2025-02-23,2025-02-23,ASG,Assigned,RHP Carlos Romero assigned to Washington Natio...,673395.0,Carlos Romero,/api/v1/people/673395,120,Washington Nationals,/api/v1/teams/120,NaN,NaN,NaN,ALL
61255,831584,2025-04-22,2025-04-22,2025-04-22,SC,Status Change,Boston Red Sox activated RHP Brayan Bello from...,678394.0,Brayan Bello,/api/v1/people/678394,111,Boston Red Sox,/api/v1/teams/111,NaN,NaN,NaN,ALL
4127,607821,2022-03-22,2022-03-22,2022-03-22,ASG,Assigned,OF Matt Pita assigned to New York Yankees.,681862.0,Matt Pita,/api/v1/people/681862,147,New York Yankees,/api/v1/teams/147,NaN,NaN,NaN,ALL
8890,621604,2022-05-17,NaT,NaN,CU,Recalled,Atlanta Braves recalled LHP Tucker Davidson fr...,656353.0,Tucker Davidson,/api/v1/people/656353,144,Atlanta Braves,/api/v1/teams/144,431.0,Gwinnett Stripers,/api/v1/teams/431,ALL
40450,739003,2024-02-01,2024-02-01,2024-02-01,ASG,Assigned,Arizona Diamondbacks invited non-roster RHP Lu...,681475.0,Luke Albright,/api/v1/people/681475,109,Arizona Diamondbacks,/api/v1/teams/109,NaN,NaN,NaN,ALL
73015,885220,2026-02-06,2026-02-06,NaN,CLW,Claimed Off Waivers,Athletics claimed 3B Andy Ibáñez off waivers f...,628451.0,Andy Ibáñez,/api/v1/people/628451,133,Athletics,/api/v1/teams/133,119.0,Los Angeles Dodgers,/api/v1/teams/119,ALL
24785,674536,2023-02-28,2023-02-28,2023-02-28,SC,Status Change,3B Kaleb Cowart roster status changed by New Y...,592230.0,Kaleb Cowart,/api/v1/people/592230,147,New York Yankees,/api/v1/teams/147,NaN,NaN,NaN,ALL
57196,817896,2025-03-02,2025-03-02,NaN,CLW,Claimed Off Waivers,Seattle Mariners claimed RHP Seth Martinez off...,661527.0,Seth Martinez,/api/v1/people/661527,136,Seattle Mariners,/api/v1/teams/136,146.0,Miami Marlins,/api/v1/teams/146,ALL
21538,665984,2023-01-17,2023-01-17,2023-01-17,ASG,Assigned,St. Louis Cardinals invited non-roster C Jimmy...,699625.0,Jimmy Crooks,/api/v1/people/699625,138,St. Louis Cardinals,/api/v1/teams/138,NaN,NaN,NaN,ALL


In [66]:
transaction_types = transactions.groupby( "type_code" ).agg(
    description=('type_desc', 'unique'),
    count=('type_desc', 'count'),
).sort_values('count', ascending=False)

In [70]:
transaction_types['description'] = transaction_types['description'].apply( lambda x: x[0] )

In [71]:
transaction_types

,description,count
type_code,,
SC,Status Change,18387
ASG,Assigned,16829
SFA,Signed as Free Agent,10194
OPT,Optioned,6272
CU,Recalled,5693
SGN,Signed,2374
SE,Selected,2373
DES,Designated for Assignment,2339
TR,Trade,2196


## Step 5: roster endpoint probes (injured / active / 40-man) for all active teams

This now iterates every currently active MLB team and pulls roster snapshots for `injured`, `active`, and `40Man` roster types on `SNAPSHOT_DATE`.


In [53]:
teams_check = fetch_json_debug("/teams", {"sportId": SPORT_ID, "season": SEASON})
team_records = payload_records(teams_check.get("payload"), ["teams"])
teams_df = pd.json_normalize(team_records) if team_records else pd.DataFrame()

if not teams_df.empty and "active" in teams_df.columns:
    active_teams = teams_df.loc[teams_df["active"] == True, ["id", "name"]].copy()
else:
    active_teams = teams_df[["id", "name"]].copy() if not teams_df.empty else pd.DataFrame(columns=["id", "name"])

active_teams = active_teams.sort_values("name").reset_index(drop=True)
print(f"Active teams selected: {len(active_teams)}")
active_teams.head(40)


Active teams selected: 30


,id,name
0,109,Arizona Diamondbacks
1,144,Atlanta Braves
2,110,Baltimore Orioles
3,111,Boston Red Sox
4,112,Chicago Cubs
5,145,Chicago White Sox
6,113,Cincinnati Reds
7,114,Cleveland Guardians
8,115,Colorado Rockies
9,116,Detroit Tigers


In [54]:
roster_probe_rows = []
roster_frames = []
roster_types = ["injured", "active", "40Man"]

for _, team in active_teams.iterrows():
    team_id = int(team["id"])
    team_name = team["name"]
    for roster_type in roster_types:
        params = {"rosterType": roster_type, "date": SNAPSHOT_DATE.isoformat()}
        result = fetch_json_debug(f"/teams/{team_id}/roster", params)
        records = payload_records(result.get("payload"), ["roster"])

        roster_probe_rows.append({
            "snapshot_date": SNAPSHOT_DATE.isoformat(),
            "team_id": team_id,
            "team_name": team_name,
            "roster_type": roster_type,
            "ok": result["ok"],
            "status": result["status"],
            "row_count": len(records),
            "url": result["url"],
            "error": result["error"],
        })

        if records:
            frame = pd.json_normalize(records)
            frame["snapshot_date"] = SNAPSHOT_DATE.isoformat()
            frame["team_id"] = team_id
            frame["team_name"] = team_name
            frame["roster_type"] = roster_type
            roster_frames.append(frame)

roster_probe_log = pd.DataFrame(roster_probe_rows)
roster_snapshot = pd.concat(roster_frames, ignore_index=True) if roster_frames else pd.DataFrame()

roster_probe_log


,snapshot_date,team_id,team_name,roster_type,ok,status,row_count,url,error
0,2026-02-22,109,Arizona Diamondbacks,injured,True,200,40,https://statsapi.mlb.com/api/v1/teams/109/rost...,None
1,2026-02-22,109,Arizona Diamondbacks,active,True,200,40,https://statsapi.mlb.com/api/v1/teams/109/rost...,None
2,2026-02-22,109,Arizona Diamondbacks,40Man,True,200,44,https://statsapi.mlb.com/api/v1/teams/109/rost...,None
3,2026-02-22,144,Atlanta Braves,injured,True,200,40,https://statsapi.mlb.com/api/v1/teams/144/rost...,None
4,2026-02-22,144,Atlanta Braves,active,True,200,40,https://statsapi.mlb.com/api/v1/teams/144/rost...,None
...,...,...,...,...,...,...,...,...,...
85,2026-02-22,141,Toronto Blue Jays,active,True,200,40,https://statsapi.mlb.com/api/v1/teams/141/rost...,None
86,2026-02-22,141,Toronto Blue Jays,40Man,True,200,41,https://statsapi.mlb.com/api/v1/teams/141/rost...,None
87,2026-02-22,120,Washington Nationals,injured,True,200,40,https://statsapi.mlb.com/api/v1/teams/120/rost...,None
88,2026-02-22,120,Washington Nationals,active,True,200,40,https://statsapi.mlb.com/api/v1/teams/120/rost...,None


In [55]:
if not roster_snapshot.empty:
    roster_view_cols = [
        c for c in [
            "team_name", "roster_type", "person.id", "person.fullName", "status.code", "status.description", "jerseyNumber", "position.abbreviation"
        ]
        if c in roster_snapshot.columns
    ]
    roster_snapshot_view = roster_snapshot[roster_view_cols].sort_values(["team_name", "roster_type", "person.fullName"])
else:
    roster_snapshot_view = pd.DataFrame()

roster_snapshot_view.head(50)


,team_name,roster_type,person.id,person.fullName,status.code,status.description,jerseyNumber,position.abbreviation
80,Arizona Diamondbacks,40Man,640462,A.J. Puk,D60,Injured 60-Day,33,P
81,Arizona Diamondbacks,40Man,680728,Adrian Del Castillo,A,Active,25,C
82,Arizona Diamondbacks,40Man,677950,Alek Thomas,A,Active,5,CF
83,Arizona Diamondbacks,40Man,694851,Andrew Hoffmann,A,Active,56,P
84,Arizona Diamondbacks,40Man,685314,Andrew Saalfrank,D60,Injured 60-Day,27,P
85,Arizona Diamondbacks,40Man,686796,Blake Walston,A,Active,48,P
86,Arizona Diamondbacks,40Man,694297,Brandon Pfaadt,A,Active,32,P
87,Arizona Diamondbacks,40Man,805299,Brandyn Garcia,A,Active,55,P
88,Arizona Diamondbacks,40Man,467793,Carlos Santana,A,Active,41,1B
89,Arizona Diamondbacks,40Man,669203,Corbin Burnes,D60,Injured 60-Day,39,P


## Step 6: normalize transactions for downstream use


In [56]:
if not transactions.empty:
    rename_map = {
        "person.id": "player_id",
        "person.fullName": "player_name",
        "toTeam.id": "to_team_id",
        "toTeam.name": "to_team_name",
        "fromTeam.id": "from_team_id",
        "fromTeam.name": "from_team_name",
        "typeCode": "type_code",
        "typeDesc": "type_desc",
        "description": "description",
        "date": "transaction_date",
        "effectiveDate": "effective_date",
    }
    transactions = transactions.rename(columns={k: v for k, v in rename_map.items() if k in transactions.columns})

    for dt_col in ["transaction_date", "effective_date"]:
        if dt_col in transactions.columns:
            transactions[dt_col] = pd.to_datetime(transactions[dt_col], errors="coerce")

transactions.head(10)


,id,transaction_date,effective_date,resolutionDate,type_code,type_desc,description,player_id,player_name,person.link,to_team_id,to_team_name,toTeam.link,from_team_id,from_team_name,fromTeam.link,requested_type_code
0,626231,2022-01-02,2022-01-02,2022-01-02,SFA,Signed as Free Agent,Minnesota Twins signed free agent RHP Jeferson...,805013.0,Jeferson Lopez,/api/v1/people/805013,142,Minnesota Twins,/api/v1/teams/142,NaN,NaN,NaN,ALL
1,602296,2022-01-03,2022-01-03,2022-01-03,SFA,Signed as Free Agent,Arizona Diamondbacks signed free agent 2B Drew...,607733.0,Drew Stankiewicz,/api/v1/people/607733,109,Arizona Diamondbacks,/api/v1/teams/109,NaN,NaN,NaN,ALL
2,605533,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Minnesota Twins signed free agent RHP Mario Sa...,532936.0,Mario Sanchez,/api/v1/people/532936,142,Minnesota Twins,/api/v1/teams/142,NaN,NaN,NaN,ALL
3,605540,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Minnesota Twins signed free agent CF Kennie Ta...,663772.0,Kennie Taylor,/api/v1/people/663772,142,Minnesota Twins,/api/v1/teams/142,NaN,NaN,NaN,ALL
4,602278,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Boston Red Sox signed free agent CF Johan Mies...,644407.0,Johan Mieses,/api/v1/people/644407,111,Boston Red Sox,/api/v1/teams/111,NaN,NaN,NaN,ALL
5,602282,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Philadelphia Phillies signed free agent 1B Ald...,625506.0,Aldrem Corredor,/api/v1/people/625506,143,Philadelphia Phillies,/api/v1/teams/143,NaN,NaN,NaN,ALL
6,602284,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Philadelphia Phillies signed free agent SS Kev...,660631.0,Kevin Vicuña,/api/v1/people/660631,143,Philadelphia Phillies,/api/v1/teams/143,NaN,NaN,NaN,ALL
7,602280,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Miami Marlins signed free agent RHP Huascar Br...,623211.0,Huascar Brazobán,/api/v1/people/623211,146,Miami Marlins,/api/v1/teams/146,NaN,NaN,NaN,ALL
8,602290,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,San Diego Padres signed free agent RHP Luis Ma...,800104.0,Luis Madrazo,/api/v1/people/800104,135,San Diego Padres,/api/v1/teams/135,NaN,NaN,NaN,ALL
9,602300,2022-01-04,2022-01-04,2022-01-04,SFA,Signed as Free Agent,Minnesota Twins signed free agent RHP Orlando ...,689620.0,Orlando Rodriguez,/api/v1/people/689620,142,Minnesota Twins,/api/v1/teams/142,NaN,NaN,NaN,ALL


## Step 7: recent fantasy-relevant dashboard


In [57]:
recent_cutoff = pd.Timestamp(date.today() - timedelta(days=LOOKBACK_DAYS))

dashboard_cols = [
    "transaction_date",
    "player_name",
    "type_desc",
    "description",
    "to_team_name",
    "from_team_name",
]
available_dashboard_cols = [c for c in dashboard_cols if c in transactions.columns]

if not transactions.empty and "transaction_date" in transactions.columns:
    recent_status = (
        transactions.loc[transactions["transaction_date"] >= recent_cutoff, available_dashboard_cols]
        .sort_values("transaction_date", ascending=False)
        .reset_index(drop=True)
    )
else:
    recent_status = pd.DataFrame(columns=available_dashboard_cols)

recent_status.head(50)


,transaction_date,player_name,type_desc,description,to_team_name,from_team_name
0,2026-02-22,Tirso Ornelas,Outrighted,San Diego Padres sent LF Tirso Ornelas outrigh...,El Paso Chihuahuas,San Diego Padres
1,2026-02-22,Ryan Nicholson,Assigned,1B Ryan Nicholson assigned to Los Angeles Angels.,Los Angeles Angels,NaN
2,2026-02-22,Ryan Harvey,Assigned,RHP Ryan Harvey assigned to Detroit Tigers.,Detroit Tigers,NaN
3,2026-02-22,Shohei Tomioka,Assigned,RHP Shohei Tomioka assigned to Athletics.,Athletics,NaN
4,2026-02-22,Sean Barnett,Assigned,CF Sean Barnett assigned to San Diego Padres.,San Diego Padres,NaN
5,2026-02-22,Thomas Ireland,Assigned,LHP Thomas Ireland assigned to Texas Rangers.,Texas Rangers,NaN
6,2026-02-22,Kane Kepley,Assigned,OF Kane Kepley assigned to Chicago Cubs.,Chicago Cubs,NaN
7,2026-02-22,Bryce Meccage,Assigned,RHP Bryce Meccage assigned to Milwaukee Brewers.,Milwaukee Brewers,NaN
8,2026-02-22,Brady Ebel,Assigned,SS Brady Ebel assigned to Milwaukee Brewers.,Milwaukee Brewers,NaN
9,2026-02-22,Luke Murphy,Assigned,RHP Luke Murphy assigned to Los Angeles Angels.,Los Angeles Angels,NaN


## Save outputs


In [59]:
transactions_out = DATA_DIR / f"player_transactions_{SEASON}.parquet"
recent_out = DATA_DIR / f"player_status_recent_{SEASON}.parquet"
call_log_out = DATA_DIR / f"player_status_call_log_{SEASON}.parquet"
roster_probe_out = DATA_DIR / f"player_status_roster_probe_{DATE_TAG}.parquet"
roster_snapshot_out = DATA_DIR / f"player_status_roster_snapshot_{DATE_TAG}.parquet"
transaction_type_out = DATA_DIR / f"player_status_transaction_types_{SEASON}.parquet"
active_teams_out = DATA_DIR / f"player_status_active_teams_{DATE_TAG}.parquet"

if not transactions.empty:
    transactions.to_parquet(transactions_out, index=False)

if not recent_status.empty:
    recent_status.to_parquet(recent_out, index=False)

if not call_log.empty:
    call_log.to_parquet(call_log_out, index=False)

if not roster_probe_log.empty:
    roster_probe_log.to_parquet(roster_probe_out, index=False)

if not roster_snapshot.empty:
    roster_snapshot.to_parquet(roster_snapshot_out, index=False)

if not transaction_type_df.empty:
    transaction_type_df.to_parquet(transaction_type_out, index=False)

if not active_teams.empty:
    active_teams.to_parquet(active_teams_out, index=False)

print("Wrote:")
for p in [
    transactions_out,
    recent_out,
    call_log_out,
    roster_probe_out,
    roster_snapshot_out,
    transaction_type_out,
    active_teams_out,
]:
    print(f" - {p}")


Wrote:
 - data/player_transactions_2023.parquet
 - data/player_status_recent_2023.parquet
 - data/player_status_call_log_2023.parquet
 - data/player_status_roster_probe_20260222.parquet
 - data/player_status_roster_snapshot_20260222.parquet
 - data/player_status_transaction_types_2023.parquet
 - data/player_status_active_teams_20260222.parquet
